In [1]:
!pip install pandas numpy requests geopandas openpyxl shapely

   ---------------------------------------- 0.0/343.3 kB ? eta -:--:--
   ---- ----------------------------------- 41.0/343.3 kB 1.9 MB/s eta 0:00:01
   --------- ------------------------------ 81.9/343.3 kB 1.1 MB/s eta 0:00:01
   --------- ------------------------------ 81.9/343.3 kB 1.1 MB/s eta 0:00:01
   ------------- ------------------------ 122.9/343.3 kB 654.9 kB/s eta 0:00:01
   --------------------- ---------------- 194.6/343.3 kB 980.4 kB/s eta 0:00:01
   ----------------------------- ---------- 256.0/343.3 kB 1.0 MB/s eta 0:00:01
   ----------------------------- ---------- 256.0/343.3 kB 1.0 MB/s eta 0:00:01
   ----------------------------- ---------- 256.0/343.3 kB 1.0 MB/s eta 0:00:01
   ----------------------------- ---------- 256.0/343.3 kB 1.0 MB/s eta 0:00:01
   ---------------------------------- --- 307.2/343.3 kB 731.4 kB/s eta 0:00:01
   -------------------------------------- 343.3/343.3 kB 760.4 kB/s eta 0:00:00
   ---------------------------------------- 0.0/1.7 


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 2 — Imports

from pathlib import Path

import pandas as pd
import numpy as np
import requests
import geopandas as gpd

from shapely.geometry import Point, Polygon

print("All libraries imported successfully.")

All libraries imported successfully.


In [3]:
# Cell 3 — Create Population folders

BASE_DIR = Path.cwd()

POPULATION_DIR = BASE_DIR / "data" / "population"

RAW_DIR = POPULATION_DIR / "raw"
PROCESSED_DIR = POPULATION_DIR / "processed"
REPORTS_DIR = POPULATION_DIR / "reports"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Population folders created successfully.")
print("Population directory:", POPULATION_DIR)

Population folders created successfully.
Population directory: d:\Courses\MachineLearning\Depi\Final Project\data\population


In [4]:
# Cell 4 — Test connection to OpenStreetMap

OSM_URL = "https://nominatim.openstreetmap.org/search"

headers = {
    "User-Agent": "UrbanMind-AI/1.0"
}

params = {
    "q": "Cairo, Egypt",
    "format": "json",
    "limit": 1
}

try:
    response = requests.get(
        OSM_URL,
        params=params,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    print("OpenStreetMap connection successful.")
    print("Results found:", len(data))

    if data:
        print("Place:", data[0]["display_name"])
        print("Latitude:", data[0]["lat"])
        print("Longitude:", data[0]["lon"])

except Exception as e:
    print("OpenStreetMap connection failed.")
    print(type(e).__name__)
    print(e)

OpenStreetMap connection successful.
Results found: 1
Place: القاهرة, مصر
Latitude: 30.0443879
Longitude: 31.2357257


In [5]:
# Cell 5 — Download Cairo administrative boundaries

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

query = """
[out:json][timeout:180];

area["name:en"="Cairo"]["boundary"="administrative"]->.cairo;

(
  relation["boundary"="administrative"]["admin_level"](area.cairo);
);

out geom;
"""

try:
    response = requests.post(
        OVERPASS_URL,
        data=query,
        headers=headers,
        timeout=240
    )

    response.raise_for_status()

    osm_data = response.json()

    print("Overpass API connection successful.")
    print("Elements downloaded:", len(osm_data.get("elements", [])))

except Exception as e:
    print("Overpass API request failed.")
    print(type(e).__name__)
    print(e)

Overpass API connection successful.
Elements downloaded: 3


In [6]:
# Cell 6 — Inspect downloaded administrative boundaries

elements = osm_data.get("elements", [])

rows = []

for element in elements:

    tags = element.get("tags", {})

    rows.append({
        "osm_id": element.get("id"),
        "name_ar": tags.get("name:ar"),
        "name_en": tags.get("name:en"),
        "name": tags.get("name"),
        "admin_level": tags.get("admin_level"),
        "boundary": tags.get("boundary"),
        "type": element.get("type")
    })

admin_df = pd.DataFrame(rows)

print("Number of administrative features:", len(admin_df))

display(admin_df)

Number of administrative features: 3


,osm_id,name_ar,name_en,name,admin_level,boundary,type
0,1473947,مصر,Egypt,مصر,2,administrative,relation
1,4103336,القاهرة,Cairo,القاهرة,4,administrative,relation
2,19526546,NaN,NaN,Janna New Cairo Compound,10,administrative,relation


In [8]:
# Cell 7 — Get smaller administrative areas inside Cairo

query_districts = """
[out:json][timeout:120];

area["name:en"="Cairo"]["boundary"="administrative"]->.cairo;

(
    relation(area.cairo)
        ["boundary"="administrative"]
        ["admin_level"~"7|8|9|10"];
);

out tags center;
"""

try:
    response = requests.post(
        OVERPASS_URL,
        data=query_districts,
        headers=headers,
        timeout=180
    )

    response.raise_for_status()

    districts_data = response.json()

    elements = districts_data.get("elements", [])

    print("Administrative areas downloaded successfully.")
    print("Number of areas found:", len(elements))

except Exception as e:
    print("Failed to download administrative areas.")
    print(type(e).__name__)
    print(e)

Administrative areas downloaded successfully.
Number of areas found: 1


In [9]:
# Cell 8 — Inspect the administrative area we found

elements = districts_data.get("elements", [])

rows = []

for element in elements:
    tags = element.get("tags", {})

    rows.append({
        "osm_id": element.get("id"),
        "name_ar": tags.get("name:ar"),
        "name_en": tags.get("name:en"),
        "name": tags.get("name"),
        "admin_level": tags.get("admin_level"),
        "boundary": tags.get("boundary"),
        "type": element.get("type"),
        "lat": element.get("center", {}).get("lat"),
        "lon": element.get("center", {}).get("lon")
    })

areas_df = pd.DataFrame(rows)

display(areas_df)

,osm_id,name_ar,name_en,name,admin_level,boundary,type,lat,lon
0,19526546,None,None,Janna New Cairo Compound,10,administrative,relation,29.954638,31.510686


In [10]:
# Cell 9 — Cairo official districts

cairo_districts = {
    "North": [
        "Shubra",
        "El-Zawia El-Hamra",
        "Hadayek El-Kobba",
        "Rod El-Farg",
        "El-Sharabia",
        "El-Sahel",
        "El-Zaiton",
        "Al-Ameria"
    ],
    
    "South": [
        "Masr El-Qadima",
        "El-Khalifa",
        "El-Moqattam",
        "El-Basatin",
        "Dar El-Salam",
        "El-Sayeda Zeinab",
        "El-Tebin",
        "Helwan",
        "El-Ma'asara",
        "El-Maadi",
        "Tora",
        "May 15"
    ],
    
    "West": [
        "Manshaet Naser",
        "El-Waily",
        "Wast El-Qahira",
        "Boulak",
        "Gharb El-Qahira",
        "Abdeen",
        "Azbakia",
        "Moski",
        "Bab El-Shaaria"
    ],
    
    "East": [
        "Misr El-Gadidah",
        "El-Nozha",
        "Sharq Madinet Nasr",
        "Gharb Madinet Nasr",
        "El-Salam Awal",
        "El-Salam Thani",
        "El-Mataria",
        "Ain Shams",
        "El-Marg"
    ]
}

district_rows = []

for zone, districts in cairo_districts.items():
    for district in districts:
        district_rows.append({
            "zone": zone,
            "district": district
        })

cairo_districts_df = pd.DataFrame(district_rows)

print("Number of Cairo districts:", len(cairo_districts_df))

display(cairo_districts_df)

Number of Cairo districts: 38


,zone,district
0,North,Shubra
1,North,El-Zawia El-Hamra
2,North,Hadayek El-Kobba
3,North,Rod El-Farg
4,North,El-Sharabia
5,North,El-Sahel
6,North,El-Zaiton
7,North,Al-Ameria
8,South,Masr El-Qadima
9,South,El-Khalifa


In [11]:
# Cell 10 — Test district geocoding with OpenStreetMap

district_name = "Shubra, Cairo, Egypt"

params = {
    "q": district_name,
    "format": "json",
    "limit": 3,
    "addressdetails": 1
}

try:
    response = requests.get(
        OSM_URL,
        params=params,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    geocode_data = response.json()

    print("Geocoding successful.")
    print("Results found:", len(geocode_data))

    for i, result in enumerate(geocode_data, start=1):
        print(f"\nResult {i}")
        print("Name:", result.get("display_name"))
        print("Latitude:", result.get("lat"))
        print("Longitude:", result.get("lon"))
        print("Type:", result.get("type"))
        print("Category:", result.get("category"))

except Exception as e:
    print("Geocoding failed.")
    print(type(e).__name__)
    print(e)

Geocoding successful.
Results found: 3

Result 1
Name: El-Mounib - Shubra El-Kheima Railway, حارة الجمعيه المارونيه, جسر شبرا, روض الفرج, القاهرة, 11627, مصر
Latitude: 30.0800201
Longitude: 31.2452690
Type: subway
Category: None

Result 2
Name: El-Mounib - Shubra El-Kheima Railway, زقاق الرحبه, الساحه, عابدين, القاهرة, 11518, مصر
Latitude: 30.0482386
Longitude: 31.2456109
Type: subway
Category: None

Result 3
Name: اتجاه شوبرا, كوبرى الليمون, الفجاله, الأزبكية, القاهرة, 11523, مصر
Latitude: 30.0669848
Longitude: 31.2451226
Type: subway
Category: None


In [12]:
# Cell 11 — Search for Shubra as an administrative boundary

district_name_ar = "شبرا"

params = {
    "q": f"حي {district_name_ar}, القاهرة, مصر",
    "format": "json",
    "limit": 10,
    "addressdetails": 1,
    "polygon_geojson": 1,
    "extratags": 1
}

try:
    response = requests.get(
        OSM_URL,
        params=params,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    geocode_data = response.json()

    print("Geocoding successful.")
    print("Results found:", len(geocode_data))

    for i, result in enumerate(geocode_data, start=1):
        print(f"\nResult {i}")
        print("Name:", result.get("display_name"))
        print("Type:", result.get("type"))
        print("Class:", result.get("class"))
        print("OSM Type:", result.get("osm_type"))
        print("OSM ID:", result.get("osm_id"))
        print("Latitude:", result.get("lat"))
        print("Longitude:", result.get("lon"))
        print("Has polygon:", bool(result.get("geojson")))

except Exception as e:
    print("Geocoding failed.")
    print(type(e).__name__)
    print(e)

Geocoding successful.
Results found: 0


In [13]:
# Cell 12 — Find Shubra administrative boundary in Cairo

query_shubra = """
[out:json][timeout:90];

area["name:en"="Cairo"]["boundary"="administrative"]->.cairo;

relation(area.cairo)
    ["boundary"="administrative"]
    ["name"~"Shubra|شبرا",i];

out tags center;
"""

try:
    response = requests.post(
        OVERPASS_URL,
        data=query_shubra,
        headers=headers,
        timeout=120
    )

    response.raise_for_status()

    shubra_data = response.json()

    elements = shubra_data.get("elements", [])

    print("Search successful.")
    print("Shubra-related administrative relations found:", len(elements))

    for element in elements:
        tags = element.get("tags", {})

        print("\nOSM ID:", element.get("id"))
        print("Arabic name:", tags.get("name:ar"))
        print("English name:", tags.get("name:en"))
        print("Name:", tags.get("name"))
        print("Admin level:", tags.get("admin_level"))
        print("Boundary:", tags.get("boundary"))

except Exception as e:
    print("Search failed.")
    print(type(e).__name__)
    print(e)

Search failed.
HTTPError
504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter


In [15]:
# Cell 12 — Search Shubra boundary using Nominatim feature lookup

params = {
    "q": "Shubra, Cairo",
    "format": "json",
    "limit": 10,
    "addressdetails": 1,
    "extratags": 1
}

try:
    response = requests.get(
        OSM_URL,
        params=params,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    results = response.json()

    print("Results found:", len(results))

    for i, result in enumerate(results, start=1):

        print(f"\n--- Result {i} ---")
        print("Display name:", result.get("display_name"))
        print("Class:", result.get("class"))
        print("Type:", result.get("type"))
        print("OSM type:", result.get("osm_type"))
        print("OSM ID:", result.get("osm_id"))
        print("Importance:", result.get("importance"))

except Exception as e:
    print("Request failed.")
    print(type(e).__name__)
    print(e)

Results found: 4

--- Result 1 ---
Display name: El-Mounib - Shubra El-Kheima Railway, حارة ابو قوطه, الفواله, باب الشعرية, القاهرة, 11518, مصر
Class: railway
Type: subway
OSM type: way
OSM ID: 981259847
Importance: 7.960402838173521e-05

--- Result 2 ---
Display name: اتجاه شوبرا, شارع جنينه الحجار, جزيره بدران, روض الفرج, القاهرة, 11644, مصر
Class: railway
Type: subway
OSM type: way
OSM ID: 45928742
Importance: 7.960402838173521e-05

--- Result 3 ---
Display name: اتجاه شوبرا, عطفه الخماره, القبيله, الأزبكية, القاهرة, 11523, مصر
Class: railway
Type: subway
OSM type: way
OSM ID: 508645789
Importance: 7.960402838173521e-05

--- Result 4 ---
Display name: محطة أتوبيس دوران شبرا, عطفه مشرف, البراد, روض الفرج, القاهرة, 11648, مصر
Class: amenity
Type: bus_station
OSM type: way
OSM ID: 60101353
Importance: 7.960402838173521e-05
